In [1]:
# ============================================================
# DAY 1 — STOCK UNIVERSE
# AI Stock Screener — Indian Markets
# Fetches NSE + BSE stock list → saves master_stocks.csv
# ============================================================

# ── CELL 1: IMPORTS ──────────────────────────────────────────
import pandas as pd
import warnings
warnings.filterwarnings('ignore')

In [4]:
# ── CELL 2: FETCH NSE STOCKS ─────────────────────────────────
nse_url = "https://archives.nseindia.com/content/equities/EQUITY_L.csv"
nse_raw = pd.read_csv(nse_url)

# Strip spaces from column names
nse_raw.columns = nse_raw.columns.str.strip()

nse_clean = pd.DataFrame({
    'Symbol'      : nse_raw['SYMBOL'].str.strip(),
    'Company Name': nse_raw['NAME OF COMPANY'].str.strip(),
    'Exchange'    : 'NSE'
})

# Keep only EQ series (regular equity shares)
nse_clean = nse_clean[
    nse_raw['SERIES'].str.strip() == 'EQ'
].reset_index(drop=True)

print(f"NSE stocks fetched: {len(nse_clean)}")

NSE stocks fetched: 2117


In [11]:
# ── CELL 3: LOAD BSE STOCKS ──────────────────────────────────
bse_raw = pd.read_csv('../data/bse_stocks.csv', index_col=False)

bse_clean = pd.DataFrame({
    'Symbol'      : bse_raw['Security Id'].str.strip(),
    'Company Name': bse_raw['Issuer Name'].str.strip(),
    'Exchange'    : 'BSE'
})

# Keep only Active Equity stocks
bse_clean = bse_clean[
    (bse_raw['Status'].str.strip() == 'Active') &
    (bse_raw['Instrument'].str.strip() == 'Equity')
].reset_index(drop=True)

# Remove BSE stocks already in NSE (avoid duplicates)
bse_clean = bse_clean[
    ~bse_clean['Symbol'].isin(nse_clean['Symbol'])
].reset_index(drop=True)

print(f"BSE stocks (Active Equity, non-duplicate): {len(bse_clean)}")
print(bse_clean.head(3).to_string(index=False))

BSE stocks (Active Equity, non-duplicate): 2842
   Symbol                           Company Name Exchange
AMBALALSA       Ambalal Sarabhai Enterprise Ltd.      BSE
ANDHRAPET             Andhra Petrochemicals Ltd.      BSE
 ANSALAPI Ansal Properties & Infrastructure Ltd.      BSE


In [12]:
# ── CELL 4: COMBINE + SAVE ───────────────────────────────────
master_list = pd.concat([nse_clean, bse_clean], ignore_index=True)
master_list.to_csv('../data/master_stocks.csv', index=False)

print(f"NSE stocks     : {len(nse_clean)}")
print(f"BSE stocks     : {len(bse_clean)}")
print(f"Total universe : {len(master_list)}")
print(f"Saved          : data/master_stocks.csv ✅")

NSE stocks     : 2117
BSE stocks     : 2842
Total universe : 4959
Saved          : data/master_stocks.csv ✅
